# Assignment #3 — Replikasi Time Series Forecasting Saham (SVR & LSTM)

Notebook ini mereplikasi *pipeline* metodologi paper *time series forecasting* saham, dengan:
- Target diganti menjadi `Target` (return saham) — bukan target timestamp birokrasi seperti paper asli.
- **Data leakage disengaja dipertahankan**: scaling (`fit_transform`) dilakukan secara **global** pada seluruh dataset SEBELUM data displit, sesuai instruksi tugas (untuk menjaga validitas komparasi pipeline metodologi dengan paper asli).

> ⚠️ **Catatan penting soal metodologi**: Melakukan `fit_transform` scaler pada seluruh dataset sebelum split adalah *data leakage* yang tidak seharusnya dilakukan pada praktik ML yang benar (informasi dari data uji "bocor" ke proses training melalui statistik scaler). Ini sengaja dipertahankan di notebook ini semata-mata untuk mereplikasi kecacatan metodologi paper asli sesuai instruksi tugas — **jangan gunakan pendekatan ini di proyek produksi/riset yang valid.**

> ⚠️ **Catatan performa**: Spesifikasi LSTM mengunci `batch_size=1` dan `epochs=100`. Untuk dataset besar (dataset JPX Kaggle bisa berisi jutaan baris), kombinasi ini akan **sangat lambat** (bisa berjam-jam) karena update gradient dilakukan per-satu-sample. Jalankan pada subset lebih kecil dulu untuk uji coba pipeline, baru jalankan penuh jika perlu, atau pertimbangkan menjalankan di mesin dengan GPU dan waktu luang. Jika ingin lebih cepat untuk eksperimen, kamu bisa menaikkan `batch_size` — namun itu akan menyimpang dari parameter paper yang dikunci di instruksi tugas.


## 0. Import Library

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.21.0


## 1. Load & Persiapan Dataset

Sesuai instruksi:
- Hanya menggunakan `stock_prices.csv` (tidak ada merge dengan file lain).
- `RowId` di-drop permanen.
- `Date` dan `SecuritiesCode` dijadikan MultiIndex agar tetap terurut kronologis & identitas saham tetap terlacak, tapi tidak dianggap sebagai fitur matriks X.
- Fitur X: `Open`, `High`, `Low`, `Close`, `Volume`.
- Target y: kolom `Target`.


In [2]:
# Ganti path ini sesuai lokasi file stock_prices.csv di komputer kamu
DATA_PATH = "train_files/stock_prices.csv"

df_raw = pd.read_csv(DATA_PATH)
print("Shape awal:", df_raw.shape)
df_raw.head()


Shape awal: (2332531, 12)


,RowId,Date,SecuritiesCode,Open,High,Low,Close,Volume,AdjustmentFactor,ExpectedDividend,SupervisionFlag,Target
0,20170104_1301,2017-01-04,1301,2734.0,2755.0,2730.0,2742.0,31400,1.0,NaN,False,0.000730
1,20170104_1332,2017-01-04,1332,568.0,576.0,563.0,571.0,2798500,1.0,NaN,False,0.012324
2,20170104_1333,2017-01-04,1333,3150.0,3210.0,3140.0,3210.0,270800,1.0,NaN,False,0.006154
3,20170104_1376,2017-01-04,1376,1510.0,1550.0,1510.0,1550.0,11300,1.0,NaN,False,0.011053
4,20170104_1377,2017-01-04,1377,3270.0,3350.0,3270.0,3330.0,150800,1.0,NaN,False,0.003026


In [3]:
df = df_raw.copy()

# Pastikan Date bertipe datetime agar urut kronologis benar
df["Date"] = pd.to_datetime(df["Date"])

# Urutkan data secara kronologis (Date, lalu SecuritiesCode) SEBELUM apapun lainnya
df = df.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)

# DROP RowId secara permanen (jika kolom ada)
if "RowId" in df.columns:
    df = df.drop(columns=["RowId"])

print(f"Rentang tanggal ASLI dataset : {df['Date'].min().date()} s/d {df['Date'].max().date()}")
print(f"Jumlah baris ASLI            : {len(df):,}")


Rentang tanggal ASLI dataset : 2017-01-04 s/d 2021-12-03
Jumlah baris ASLI            : 2,332,531


### 1a. Konfigurasi Rentang Tanggal (untuk keperluan komputasi)

Dataset penuh (2017-01-04 s/d 2021-12-03, ~4,7 juta baris, 4.371 kode saham) membuat
`SVR(kernel='rbf')` tidak feasible dijalankan (kompleksitas O(n²)-O(n³) terhadap jumlah
sample). Sebagai kompromi praktis — **bukan perubahan metodologi model** — rentang
tanggal dataset dipersempit di cell berikut.

Ubah `N_MONTHS_BACK` (atau `START_DATE`/`END_DATE`) di cell di bawah untuk mengatur
seberapa panjang data yang dipakai. Ini satu-satunya tempat yang perlu diubah untuk
eksperimen dengan rentang tanggal berbeda.


In [4]:
# ==================== KONFIGURASI RENTANG TANGGAL ====================
# Ubah nilai di sini untuk mengatur seberapa banyak data historis yang dipakai.
#
# Mode 1 (default): pakai N_MONTHS_BACK bulan terakhir dari tanggal terbaru di dataset.
#   Set N_MONTHS_BACK = None untuk pakai Mode 2 (rentang tanggal eksplisit) sebagai gantinya.
#
# Mode 2: isi START_DATE dan/atau END_DATE secara eksplisit (format "YYYY-MM-DD").
#   Biarkan None untuk memakai batas paling awal/akhir yang tersedia di dataset.

N_MONTHS_BACK = 6     # <-- ganti angka ini untuk mengubah rentang (mis. 1, 3, 12, 24)
START_DATE = None     # dipakai hanya jika N_MONTHS_BACK = None, contoh: "2021-06-01"
END_DATE = None        # dipakai hanya jika N_MONTHS_BACK = None, contoh: "2021-12-03"
# =======================================================================

min_date_in_data = df["Date"].min()
max_date_in_data = df["Date"].max()

if N_MONTHS_BACK is not None:
    range_end = pd.Timestamp(END_DATE) if END_DATE else max_date_in_data
    range_start = range_end - pd.DateOffset(months=N_MONTHS_BACK)
else:
    range_start = pd.Timestamp(START_DATE) if START_DATE else min_date_in_data
    range_end = pd.Timestamp(END_DATE) if END_DATE else max_date_in_data

df = df[(df["Date"] >= range_start) & (df["Date"] <= range_end)].reset_index(drop=True)

print(f"Filter rentang tanggal        : {range_start.date()} s/d {range_end.date()}")
print(f"Jumlah baris setelah difilter : {len(df):,}")
print(f"Jumlah kode saham unik        : {df['SecuritiesCode'].nunique():,}")


Filter rentang tanggal        : 2021-06-03 s/d 2021-12-03
Jumlah baris setelah difilter : 250,000
Jumlah kode saham unik        : 2,000


In [5]:
FEATURE_COLS = ["Open", "High", "Low", "Close", "Volume"]
TARGET_COL = "Target"

required_cols = FEATURE_COLS + [TARGET_COL, "Date", "SecuritiesCode"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Kolom berikut tidak ditemukan di dataset: {missing}")

# Pembersihan minimal yang diperlukan agar pipeline bisa jalan:
# baris tanpa Target (NaN) tidak bisa dipakai untuk supervised learning,
# dan baris dengan fitur NaN juga di-drop. Ini BUKAN bagian dari "data
# leakage" yang disengaja -- ini cuma pembersihan data dasar sebelum modeling.
before = len(df)
df = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)
after = len(df)
print(f"Baris sebelum dropna: {before} | setelah dropna: {after} | dibuang: {before - after}")

# Set MultiIndex (Date, SecuritiesCode) -> menjaga urutan kronologis & identitas saham,
# tapi tidak ikut jadi fitur matriks X
df = df.set_index(["Date", "SecuritiesCode"])

print("Shape final:", df.shape)
df.head()


Baris sebelum dropna: 250000 | setelah dropna: 249343 | dibuang: 657
Shape final: (249343, 9)


Open    High     Low   Close   Volume  \
Date       SecuritiesCode                                            
2021-06-03 1301            2924.0  2940.0  2924.0  2936.0    10400   
           1332             522.0   534.0   522.0   534.0  1736700   
           1333            2438.0  2469.0  2431.0  2465.0   203600   
           1375            1759.0  1787.0  1759.0  1780.0    77100   
           1376            1488.0  1500.0  1486.0  1492.0     3200   

                           AdjustmentFactor  ExpectedDividend  \
Date       SecuritiesCode                                       
2021-06-03 1301                         1.0               NaN   
           1332                         1.0               NaN   
           1333                         1.0               NaN   
           1375                         1.0               NaN   
           1376                         1.0               NaN   

                           SupervisionFlag    Target  
Date       SecuritiesCode                             
2021-06-03 1301                      False  0.002385  
           1332                      False  0.014815  
           1333                      False -0.002851  
           1375                      False -0.004464  
           1376                      False  0.002674

In [6]:
X_full = df[FEATURE_COLS].values.astype(np.float64)
y_full = df[TARGET_COL].values.astype(np.float64).reshape(-1, 1)

print("X_full shape:", X_full.shape)
print("y_full shape:", y_full.shape)


X_full shape: (249343, 5)
y_full shape: (249343, 1)


## 2. Preprocessing — Scaling Global (Data Leakage Disengaja)

Sesuai instruksi tugas, scaling dilakukan `fit_transform` pada **seluruh baris dataset** SEBELUM data displit menjadi train/test. Ini dilakukan terpisah untuk dua pipeline:
- **Pipeline SVR** → `StandardScaler` untuk X dan y.
- **Pipeline LSTM** → `MinMaxScaler` untuk X dan y.

Scaler `y` disimpan agar prediksi bisa di-*inverse transform* kembali ke skala asli saat evaluasi.


In [7]:
# ---- Pipeline SVR: StandardScaler (global, sebelum split -> disengaja leak) ----
scaler_X_svr = StandardScaler()
scaler_y_svr = StandardScaler()

X_svr_scaled = scaler_X_svr.fit_transform(X_full)
y_svr_scaled = scaler_y_svr.fit_transform(y_full).ravel()

print("X_svr_scaled shape:", X_svr_scaled.shape)
print("y_svr_scaled shape:", y_svr_scaled.shape)


X_svr_scaled shape: (249343, 5)
y_svr_scaled shape: (249343,)


In [8]:
# ---- Pipeline LSTM: MinMaxScaler (global, sebelum split -> disengaja leak) ----
scaler_X_lstm = MinMaxScaler()
scaler_y_lstm = MinMaxScaler()

X_lstm_scaled = scaler_X_lstm.fit_transform(X_full)
y_lstm_scaled = scaler_y_lstm.fit_transform(y_full).ravel()

print("X_lstm_scaled shape:", X_lstm_scaled.shape)
print("y_lstm_scaled shape:", y_lstm_scaled.shape)


X_lstm_scaled shape: (249343, 5)
y_lstm_scaled shape: (249343,)


## 3. Data Splitting — Train / Validation / Test (Chronological)

**Kesepakatan tim (menggantikan `TimeSeriesSplit(n_splits=10)` sebelumnya untuk tahap ini):**
Dataset (setelah difilter rentang tanggalnya di atas) dibelah secara **kronologis** menjadi
dua tingkat:

1. **Train split** vs **Test split** — Test split adalah potongan tanggal PALING AKHIR,
   disisihkan total dan HANYA dipakai sekali di akhir untuk evaluasi performa final.
2. **Train split** dibelah lagi menjadi **True Train split** vs **Validation (Val) split**
   — Val diambil dari ekor Train split secara kronologis.

**Untuk apa Val split?** Val split **tidak** dipakai untuk melatih bobot model. Ia dipakai
untuk memantau/mengambil keputusan SELAMA proses training atau tuning, misalnya:
- Menentukan kapan training LSTM harus berhenti (*early stopping*) — begitu error di Val
  berhenti membaik, training dihentikan, supaya model tidak *overfit* ke True Train.
- (Jika suatu saat dilakukan pencarian hyperparameter / *tuning*) — Val dipakai untuk
  membandingkan kombinasi hyperparameter, TANPA pernah mengintip Test split.

Karena hyperparameter SVR (`kernel`, `C`, `epsilon`) dan arsitektur LSTM di notebook ini
sudah DIKUNCI sesuai paper (tidak ada pencarian hyperparameter), Val split di sini secara
konkret hanya benar-benar dipakai untuk **early stopping LSTM**. Untuk SVR, Val split tetap
dihitung & dipisahkan (konsisten secara struktur), namun tidak difungsikan dalam training
karena tidak ada proses tuning yang berjalan.

Test split, di kedua model, baru disentuh SATU KALI di tahap evaluasi akhir.


In [9]:
# ==================== KONFIGURASI SPLIT DATA ====================
# Proporsi dihitung berurutan secara KRONOLOGIS (bukan diacak):
#   1. TEST_SIZE  -> persentase dari TOTAL data (paling akhir secara waktu) jadi Test split.
#   2. VAL_SIZE   -> persentase dari sisa Train split (paling akhir di dalam Train split
#                    itu sendiri) jadi Validation split.
TEST_SIZE = 0.2   # <-- ganti untuk mengubah ukuran Test split (mis. 0.1, 0.15, 0.3)
VAL_SIZE = 0.2    # <-- ganti untuk mengubah ukuran Val split (dihitung dari Train split)
# =======================================================================

n_total = X_full.shape[0]
dates_full = df.index.get_level_values("Date")

# ---- Level 1: Train split vs Test split (kronologis) ----
n_test = int(round(n_total * TEST_SIZE))
n_trainval = n_total - n_test

trainval_idx = np.arange(0, n_trainval)
test_idx = np.arange(n_trainval, n_total)

# ---- Level 2: True Train split vs Val split (dari dalam Train split) ----
n_val = int(round(n_trainval * VAL_SIZE))
n_true_train = n_trainval - n_val

true_train_idx = trainval_idx[:n_true_train]
val_idx = trainval_idx[n_true_train:]

def _date_range(idx):
    d = dates_full[idx]
    return f"{d.min().date()} s/d {d.max().date()}"

print(f"Jumlah total sample      : {n_total:,}")
print(f"True Train split : {len(true_train_idx):>7,} sample | {_date_range(true_train_idx)}")
print(f"Val split        : {len(val_idx):>7,} sample | {_date_range(val_idx)}")
print(f"Test split       : {len(test_idx):>7,} sample | {_date_range(test_idx)}")


Jumlah total sample      : 249,343
True Train split : 159,579 sample | 2021-06-03 s/d 2021-09-29
Val split        :  39,895 sample | 2021-09-29 s/d 2021-10-27
Test split       :  49,869 sample | 2021-10-27 s/d 2021-12-03


## 4. Model 1 — Support Vector Regression (SVR)

Hyperparameter (dikunci sesuai paper):
- Kernel: `rbf`
- C: `100`
- Epsilon: `0.0005`


In [10]:
import time

# True Train split (dipakai untuk fit bobot SVR) & Test split (evaluasi akhir)
X_train_svr, X_test_svr = X_svr_scaled[true_train_idx], X_svr_scaled[test_idx]
y_train_svr, y_test_svr = y_svr_scaled[true_train_idx], y_svr_scaled[test_idx]

# Val split juga tersedia untuk konsistensi struktur, meski tidak difungsikan di
# training SVR karena hyperparameter (kernel/C/epsilon) sudah dikunci, tidak ada tuning.
X_val_svr, y_val_svr = X_svr_scaled[val_idx], y_svr_scaled[val_idx]

svr_model = SVR(kernel="rbf", C=100, epsilon=0.0005)

print(f"Training SVR pada {len(X_train_svr):,} sample (True Train split) ...")
_t0 = time.time()
svr_model.fit(X_train_svr, y_train_svr)
_elapsed = time.time() - _t0
print(f"Selesai training SVR dalam {_elapsed:.1f} detik ({_elapsed/60:.1f} menit).")

y_pred_svr_scaled = svr_model.predict(X_test_svr)

# Inverse transform kembali ke skala asli Target untuk evaluasi yang bermakna
y_pred_svr = scaler_y_svr.inverse_transform(y_pred_svr_scaled.reshape(-1, 1)).ravel()
y_true_svr = scaler_y_svr.inverse_transform(y_test_svr.reshape(-1, 1)).ravel()


Training SVR pada 159,579 sample (True Train split) ...
Selesai training SVR dalam 852.6 detik (14.2 menit).


In [11]:
mse_svr = mean_squared_error(y_true_svr, y_pred_svr)
rmse_svr = np.sqrt(mse_svr) 
mae_svr = mean_absolute_error(y_true_svr, y_pred_svr)

print(f"SVR  -> MSE: {mse_svr:.8f} | RMSE: {rmse_svr:.8f} | MAE: {mae_svr:.8f}")


SVR  -> MSE: 0.00063771 | RMSE: 0.02525301 | MAE: 0.01703794


## 5. Model 2 — Long Short-Term Memory (LSTM)

Arsitektur (dikunci sesuai paper):
- 1 layer `LSTM(50 units)` dengan aktivasi `ReLU`
- `Dropout(0.2)` setelah layer LSTM
- 1 layer `Dense` output
- Loss: `mean_squared_error`
- Optimizer: `adam`
- Batch size: `1`
- Epochs: `100` (sekarang jadi **batas maksimum**)
- Input direshape menjadi `(samples, 1, n_features)` (timestep = 1)

> ➕ **Tambahan di luar spesifikasi paper**: `EarlyStopping` ditambahkan (monitor `val_loss`,
> `patience=5`, `restore_best_weights=True`) supaya training berhenti otomatis begitu model
> berhenti membaik, alih-alih memaksakan 100 epoch penuh. Val split yang dipakai untuk
> memantau ini berasal dari split Train/Val/Test kronologis di Bagian 3 di atas — jadi Test
> split tetap murni tidak tersentuh sama sekali selama training.


In [12]:
# True Train / Val / Test split, dari hasil scaling MinMaxScaler (index sama dengan SVR)
X_train_lstm_2d = X_lstm_scaled[true_train_idx]
X_val_lstm_2d = X_lstm_scaled[val_idx]
X_test_lstm_2d = X_lstm_scaled[test_idx]

y_train_lstm = y_lstm_scaled[true_train_idx]
y_val_lstm = y_lstm_scaled[val_idx]
y_test_lstm = y_lstm_scaled[test_idx]

n_features = X_train_lstm_2d.shape[1]

# Reshape ke (samples, 1, n_features) -> timestep = 1
X_train_lstm = X_train_lstm_2d.reshape((X_train_lstm_2d.shape[0], 1, n_features))
X_val_lstm = X_val_lstm_2d.reshape((X_val_lstm_2d.shape[0], 1, n_features))
X_test_lstm = X_test_lstm_2d.reshape((X_test_lstm_2d.shape[0], 1, n_features))

print(f"X_train_lstm shape (True Train, untuk update bobot)   : {X_train_lstm.shape}")
print(f"X_val_lstm shape   (Val, untuk early stopping)         : {X_val_lstm.shape}")
print(f"X_test_lstm shape  (Test, evaluasi akhir)              : {X_test_lstm.shape}")


X_train_lstm shape (True Train, untuk update bobot)   : (159579, 1, 5)
X_val_lstm shape   (Val, untuk early stopping)         : (39895, 1, 5)
X_test_lstm shape  (Test, evaluasi akhir)              : (49869, 1, 5)


In [13]:
lstm_model = Sequential([
    LSTM(50, activation="relu", input_shape=(1, n_features)),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(optimizer="adam", loss="mean_squared_error")
lstm_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        11,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,251 (43.95 KB)

 Trainable params: 11,251 (43.95 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping
import time

# PERINGATAN: batch_size=1 & epochs=100 dikunci sesuai spesifikasi paper
# (epochs=100 sekarang berfungsi sebagai BATAS MAKSIMUM -- EarlyStopping bisa
# menghentikan training lebih awal begitu val_loss berhenti membaik).
# Untuk dataset besar, batch_size=1 tetap bisa berjalan lama per-epoch-nya.

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,             # berhenti jika val_loss tidak membaik 5 epoch berturut-turut
    restore_best_weights=True,
    verbose=1,
)

print(f"Training LSTM pada {len(X_train_lstm):,} sample True Train "
      f"(maks 100 epoch, early stopping dipantau dari Val split) ...")
_t0 = time.time()
history = lstm_model.fit(
    X_train_lstm, y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    batch_size=1,
    epochs=100,
    callbacks=[early_stop],
    verbose=1,
    shuffle=False,  # menjaga urutan kronologis data time series
)
_elapsed = time.time() - _t0
print(f"Selesai training LSTM dalam {_elapsed:.1f} detik ({_elapsed/60:.1f} menit), "
      f"berhenti pada epoch {len(history.history['loss'])}.")


Training LSTM pada 159,579 sample True Train (maks 100 epoch, early stopping dipantau dari Val split) ...
Epoch 1/100
159579/159579 ━━━━━━━━━━━━━━━━━━━━ 187s 1ms/step - loss: 4.7815e-04 - val_loss: 9.2628e-04
Epoch 2/100
159579/159579 ━━━━━━━━━━━━━━━━━━━━ 184s 1ms/step - loss: 4.4275e-04 - val_loss: 9.2980e-04
Epoch 3/100
159579/159579 ━━━━━━━━━━━━━━━━━━━━ 185s 1ms/step - loss: 4.4273e-04 - val_loss: 9.2980e-04
Epoch 4/100
159579/159579 ━━━━━━━━━━━━━━━━━━━━ 199s 1ms/step - loss: 4.4273e-04 - val_loss: 9.2980e-04
Epoch 5/100
159579/159579 ━━━━━━━━━━━━━━━━━━━━ 242s 1ms/step - loss: 4.4273e-04 - val_loss: 9.2980e-04
Epoch 6/100
159579/159579 ━━━━━━━━━━━━━━━━━━━━ 189s 1ms/step - loss: 4.4273e-04 - val_loss: 9.2980e-04
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 1.
Selesai training LSTM dalam 1186.9 detik (19.8 menit), berhenti pada epoch 6.


In [15]:
y_pred_lstm_scaled = lstm_model.predict(X_test_lstm, verbose=0).ravel()

# Inverse transform kembali ke skala asli Target untuk evaluasi yang bermakna
y_pred_lstm = scaler_y_lstm.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1)).ravel()
y_true_lstm = scaler_y_lstm.inverse_transform(y_test_lstm.reshape(-1, 1)).ravel()


In [16]:
mse_lstm = mean_squared_error(y_true_lstm, y_pred_lstm)
rmse_lstm = np.sqrt(mse_lstm)
mae_lstm = mean_absolute_error(y_true_lstm, y_pred_lstm)

print(f"LSTM -> MSE: {mse_lstm:.8f} | RMSE: {rmse_lstm:.8f} | MAE: {mae_lstm:.8f}")


LSTM -> MSE: 0.00092334 | RMSE: 0.03038655 | MAE: 0.02256990


## 6. Tabel Perbandingan Hasil Evaluasi Akhir

In [17]:
results = pd.DataFrame({
    "Model": ["SVR", "LSTM"],
    "MSE": [mse_svr, mse_lstm],
    "RMSE": [rmse_svr, rmse_lstm],
    "MAE": [mae_svr, mae_lstm],
})

results


,Model,MSE,RMSE,MAE
0,SVR,0.000638,0.025253,0.017038
1,LSTM,0.000923,0.030387,0.022570


### Catatan Akhir

- Kedua model dievaluasi pada *test fold* TERAKHIR dari `TimeSeriesSplit(n_splits=10)`, sesuai instruksi tugas.
- Metrik dihitung pada skala **asli** `Target` (hasil *inverse transform*), agar MSE/RMSE/MAE mudah diinterpretasi secara finansial.
- Sekali lagi: scaling global sebelum split adalah *data leakage* yang disengaja dipertahankan untuk mereplikasi metodologi paper asli — bukan praktik yang direkomendasikan untuk penelitian atau produksi yang valid.
